```mermaid
graph TD
    Start((程序启动)) --> Main[执行 main 函数]
    
    subgraph 异常捕获层
        Main --> Catch{捕获异常?}
        Catch -- KeyboardInterrupt --> KI[记录警告: 用户手动中断 Ctrl+C] --> Exit((结束))
        Catch -- Exception --> EX[记录严重错误: 未捕获的致命崩溃] --> Exit
    end

    Catch -- 正常执行 --> ParseArgs[解析命令行参数 parse_args]
    ParseArgs --> SetupLog[初始化双通道日志 setup_logging]
    
    SetupLog --> CheckMaint{是否触发资产维护命令?<br>handle_maintenance}
    
    subgraph 维护模式分支
        CheckMaint -- "--query-backup" --> MQ[查询备份资产状态] --> ExitMaint[退出程序]
        CheckMaint -- "--backup" --> MB[执行手动备份] --> ExitMaint
        CheckMaint -- "--restore" --> MR[从备份恢复数据] --> ExitMaint
        ExitMaint --> Exit
    end
    
    subgraph 核心管线准备与执行
        CheckMaint -- 否 (常规分析模式) --> ValidateCluster{验证星团参数 cluster}
        ValidateCluster -- 传入 'all' --> C_All[提取所有配置的星团ID]
        ValidateCluster -- 单个或多个逗号分隔 --> C_Valid[逐个验证并标准化星团名称 _validate_cluster]
        
        C_All --> ParseModes[确定特征空间 feature_spaces<br>处理 'all' 或单模式]
        C_Valid --> ParseModes
        
        ParseModes --> ParseKVP[解析微调参数字典<br>algo, audit, seed _parse_key_value_pairs]
        ParseKVP --> InitWF[实例化 AstroWorkflow]
        InitWF --> RunWF[调用 wf.run 委托执行批处理]
        RunWF --> Exit
    end

```mermaid
flowchart TD
    Start((🚀 Pipeline Start)) --> InitWF["初始化 AstroWorkflow<br/>连接 AstroDB"]
    InitWF --> BatchRun["调用 run 方法<br/>输入: clusters, categories, modes, algos..."]
    
    subgraph BatchLoops ["Cartesian Product Loops"]
        direction TB
        LoopCluster["For each cluster_id"] --> LoopCat["For each category"]
        
        subgraph SharedPrep ["共享数据准备 (Once per Cluster/Cat)"]
            PrepData["Phase 1: _prepare_shared_data<br/>导入原始数据, 标准化参考表"]
        end
        LoopCat --> SharedPrep
        SharedPrep --> LoopMode["For each feature_space"]
        
        LoopMode --> LoopAlgo["For each algorithm"]
        
        subgraph SingleExec ["单次管线执行"]
            RunSingle["调用 _execute_single_pipeline<br/>skip_data_prep=True"]
            RunSingle -.-> CatchExc{"捕获异常?"}
            CatchExc -- Yes --> LogErr["记录错误日志"]
            CatchExc -- No --> CollectRes["收集 summary 结果"]
        end
        LoopAlgo --> SingleExec
        
        CollectRes --> NextAlgo["Next Algorithm"]
        LogErr --> NextAlgo
        NextAlgo --> LoopAlgo
        
        NextAlgo --End Loop--> NextMode["Next Mode"]
        NextMode --> LoopMode
        
        NextMode --End Loop--> NextCat["Next Category"]
        NextCat --> LoopCat
        
        NextCat --End Loop--> NextCluster["Next Cluster"]
        NextCluster --> LoopCluster
    end
    
    BatchRun --> BatchLoops
    BatchLoops --End All--> SummaryRep["渲染批量汇总报告"]
    SummaryRep --> UnionView["注册跨星团联合视图"]
    UnionView --> End((🏁 Finished))

```mermaid
flowchart TD
    direction TB
    Start((Start Single Pipeline)) --> ContextIn["接收 RunContext"]

    subgraph Phase1 ["Phase 1: 数据准备 & 上下文确定"]
        DecidePrep{"skip_data_prep?"}
        DecidePrep -- No --> StepPrepShared["执行 _prepare_shared_data<br/>建立领域模型, 标准化参考表"]
        DecidePrep -- Yes --> StepFinalizeCtx["执行 _finalize_context<br/>确定 GMM 配置, 特征列表, Master表名"]
        StepPrepShared --> StepFinalizeCtx
    end
    Phase1 --> Phase2

    subgraph Phase2 ["Phase 2: GMM 成员识别 _compute_members"]
        direction TB
        CheckCache{"是否有缓存?"}
        CheckCache -- Yes --> LoadCache["加载缓存结果表"]
        CheckCache -- No --> LoadField

        subgraph DataLoad ["数据加载与变换"]
            LoadField["_load_and_transform_field<br/>Field数据 + 特征变换 + NaN清洗"]
            InitMaster["初始化数据库 Master 表"]
            LoadSeeds["_load_and_transform_seeds<br/>Seed数据 + 特征变换 + NaN清洗 + 标签回写"]
            LoadField --> InitMaster --> LoadSeeds
        end
        
        DataLoad --> DecidePath{"use_experimental?"}
        
        subgraph StablePath ["稳定生产轨"]
            PriorGMM["运行 PriorGMM <br/>Fit & Predict"]
        end
        
        subgraph ExperiPath ["实验新轨 _run_experimental_pipeline"]
            direction TB
            SeedRefine["种子精炼: ClusterSeedExtractor<br/>DBSCAN降噪, 标记 refined_seed"]
            StrategyInit["实例化消歧引擎策略<br/>Bayesian/Threshold/Blind"]
            
            ChannelA["🎯 通道 A: 抓取核心成员<br/>engine.fit_predict"]
            
            subgraph ChannelB ["📐 通道 B: 潮汐尾捕捉"]
                AdaptTube["自适应计算管尺寸<br/>Hard Cap 防护"]
                BuildTube["_build_spatial_tube<br/> Tangential投影, PCA/自行引导主轴"]
                SigmaClip["运动学 Sigma Clip<br/>马氏距离剔除 outliers"]
                TubeTag["回灌 Master 表:<br/>in_tube, pca坐标"]
                TailSeed["构建 Tail 种子模板<br/>去核管星"]
                TailCompute["管内 Tail 拟合与预测"]
                
                AdaptTube --> BuildTube --> SigmaClip --> TubeTag --> TailSeed --> TailCompute
            end
            
            Fusion["⚖️ 分层决策融合 Hierarchical Union<br/>Core优先, Tail接管, 标记source"]
            
            SeedRefine --> StrategyInit --> ChannelA --> ChannelB --> Fusion
        end
        
        DecidePath -- No --> StablePath
        DecidePath -- Yes --> ExperiPath
        
        StablePath --> TagMaster["回灌概率及分通道信息至 Master 表"]
        ExperiPath --> TagMaster
        LoadCache --> TagMaster
    end
    Phase2 --> Phase3

    subgraph Phase3 ["Phase 3: 后处理 _post_process"]
        MarkFlags["标记 is_golden, is_candidate<br/>基于概率门限"]
        CalcStats["计算统计计数<br/>Raw/Clean/Refined/Golden/Candidate"]
        RegView["注册候选者视图 v_candidates"]
        MarkFlags --> CalcStats --> RegView
    end
    Phase3 --> Phase4

    subgraph Phase4 ["Phase 4: 审计 _audit_phase"]
        direction TB
        CrossMatch["_run_cross_match<br/>PG vs Ref 交叉比对<br/>标记 Matched/PG Only/Ref Only"]
        
        subgraph DeepAudit ["深度审计 Validator.run"]
            WarmUp["SIMBAD 缓存预热<br/>增量网络同步"]
            PhysAudit["物理一致性审计<br/>卡方/惩罚项"]
            LitAudit["文献共识审计<br/>SIMBAD 语义树匹配"]
            DecideFusion["融合决策<br/>物理 × 文献 → audit_status"]
            WarmUp --> PhysAudit --> LitAudit --> DecideFusion
        end
        
        CrossMatch --> SubsetLoop["For each subset:<br/>PG_Only, Ref_Only, Matched"]
        SubsetLoop --> DeepAudit
        DeepAudit --> SyncMaster["将审计结果同步至 Master 表"]
        SyncMaster --> SubsetLoop
        SubsetLoop --End--> CombineStats["组合集统计相加"]
    end
    Phase4 --> Phase5

    subgraph Phase5 ["Phase 5: 导出 _export_phase"]
        Detailed{"result_mode == 'detailed'?"}
        Detailed -- Yes --> ExportCSV["导出 Master表 & 审计视图至 CSV"]
        Detailed -- No --> SkipExport["⏩ 跳过导出"]
    end
    Phase5 --> Phase6

    subgraph Phase6 ["Phase 6: 报告与可视化"]
        AstroAnalyzer["运行 AstroAnalyzer.run_phase6<br/>生成报告和图表"]
    end
    Phase6 --> EndSingle((End & Return Summary))


```mermaid
flowchart TD
    direction TB

    A["获取 Gaia 数据库星表<br>指定中心坐标与半径"] --> B["数据预处理<br>剔除 RA/Dec/PM/Plx 缺失野星"]

    subgraph core_channel ["核心通道 Core Channel"]
        B --> C["DBSCAN 5D 聚类<br>提取最大簇作为核心种子"]
        C --> D["GMM 模型拟合<br>训练背景与星团高斯模型"]
        D --> E["贝叶斯 EMA 迭代<br>动态更新权重 f, 计算 P_core"]
    end

    subgraph tail_channel ["潮汐尾通道 Tidal Tail Channel"]
        E --> F["PCA 空间几何掩模<br>切割潮汐管 Tidal Tube"]
        F --> G["提取潮汐种子<br>剔除 P_core > 0.9 的高置信星"]
        G --> H["二次 GMM 拟合<br>训练潮汐尾局部模型"]
        H --> I["贝叶斯独立推理<br>计算 P_tail"]
    end

    subgraph union_verify ["决策融合与验证 Decision & Verification"]
        E --> J{"概率分层决策树"}
        I --> J
        
        J -->|"P_core > 0.5"| K["核心成员<br>Core Member"]
        J -->|"P_core < 0.5 且 P_tail > 0.5"| L["潮汐尾成员<br>Tidal Tail Member"]
        J -->|"其它"| M["背景场星<br>Field Star"]
        
        K -.-> N["独立算法对比验证<br>输入运动学数据至 pyUPMASK"]
        L -.-> N
        M -.-> N
    end

    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef highlight fill:#e1f5fe,stroke:#0288d1,stroke-width:2px;
    class C,D,E,F,G,H,I,J highlight;